# AquaCast EDA Replacements Version 13
This notebook generates the EDA replacement figures used in the edited paper.


In [ ]:
# AquaCast EDA replacement figures - Version 13
# This notebook creates replacement EDA figures for the research paper.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("/content/drive/MyDrive/Datasets/Aquacast15Years_Weekly.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/Datasets/AquaCast_EDA_Replacements_Version_13")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")
for col in ["result_value", "rain_3day_sum", "temp_3day_avg", "target_exceed"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

ecoli = df[df["indicator_clean"].eq("E. coli")].copy()
enterococcus = df[df["indicator_clean"].eq("Enterococcus")].copy()

# -----------------------------
# Fig 4: Class balance / exceedance rate
# -----------------------------
summary = []
for name, sub in [("E. coli", ecoli), ("Enterococcus", enterococcus)]:
    total = sub["target_exceed"].notna().sum()
    exceedance = int(sub["target_exceed"].fillna(0).sum())
    below = total - exceedance
    summary.append({
        "bacteria": name,
        "below_threshold": below,
        "exceedance": exceedance,
        "total": total,
        "exceedance_rate": exceedance / total * 100 if total else np.nan,
    })
summary_df = pd.DataFrame(summary)
display(summary_df)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
x = np.arange(len(summary_df))
axes[0].bar(x - 0.18, summary_df["below_threshold"], width=0.36, label="Below threshold")
axes[0].bar(x + 0.18, summary_df["exceedance"], width=0.36, label="Exceedance")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["bacteria"])
axes[0].set_ylabel("Record count")
axes[0].set_title("Class counts")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(summary_df["bacteria"], summary_df["exceedance_rate"])
axes[1].set_ylabel("Exceedance rate (%)")
axes[1].set_title("Exceedance rate")
axes[1].set_ylim(0, max(60, summary_df["exceedance_rate"].max() * 1.25))
axes[1].grid(axis="y", alpha=0.25)
for i, v in enumerate(summary_df["exceedance_rate"]):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
fig.suptitle("EDA: threshold-exceedance class balance", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig4_eda_class_balance.png", dpi=220, bbox_inches="tight")
plt.show()

# -----------------------------
# Helper for scatter plots
# -----------------------------
def make_scatter_pair(xcol, xlabel, title, outfile):
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
    for ax, (name, sub, threshold) in zip(
        axes,
        [("E. coli", ecoli, 235), ("Enterococcus", enterococcus, 130)]
    ):
        plot_df = sub[[xcol, "result_value"]].dropna()
        ax.scatter(plot_df[xcol], plot_df["result_value"], s=18, alpha=0.75)
        ax.axhline(threshold, linestyle="--", linewidth=1.2, label=f"Threshold = {threshold}")
        ax.set_title(name)
        ax.set_xlabel(xlabel)
        ax.set_yscale("log")
        ax.set_ylabel("FIB result (MPN/100 mL)")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / outfile, dpi=220, bbox_inches="tight")
    plt.show()

# Fig 5-6: rainfall EDA
make_scatter_pair(
    "rain_3day_sum",
    "3-day rain sum",
    "EDA: rainfall and measured bacteria results",
    "fig5_6_eda_rain_scatter.png"
)

# Fig 7-8: temperature EDA
make_scatter_pair(
    "temp_3day_avg",
    "3-day average temperature (C)",
    "EDA: temperature and measured bacteria results",
    "fig7_8_eda_temperature_scatter.png"
)